# DSPy

A refresher on **DSPy** — a framework for *programming* language models instead of
*prompting* them. You declare the **signature** of each LM step (typed inputs → typed
outputs), compose steps as ordinary Python **modules**, and let an **optimizer** write and
tune the actual prompts (and few-shot demos) for you against a metric.

**Domain:** LLM Inference, Training & Optimization  ·  **runnable:** yes (offline, via `DummyLM` — no API key needed)

## 1. What & Why

**The problem:** prompt engineering is brittle string-wrangling. You hand-tune a wall of text,
it works on `gpt-4o`, then you swap models or change the task and the magic words stop working.
There's no separation between *what* you want and *how* you ask for it, and no principled way to
improve a prompt other than fiddling.

**DSPy's answer:** treat an LLM pipeline like a program you *compile*, not a prompt you *write*.

- You declare **signatures** — `question -> answer`, with field names, types, and short
  descriptions — and pick a **module** (`Predict`, `ChainOfThought`, `ReAct`, …) that implements
  the prompting strategy.
- DSPy turns that into the actual prompt at run time via an **adapter** (the chat-format template
  with `[[ ## field ## ]]` markers).
- An **optimizer** (a.k.a. teleprompter) then *searches* over instructions and few-shot examples,
  scoring candidates with your **metric** on a small trainset, and bakes the winners back into the
  program. Same Python code, much better prompt — and you can re-optimize when you switch models.

**Reach for DSPy when** you have a multi-step LLM pipeline (RAG, agents, classification,
extraction) that you want to *measure and improve systematically*, or when you need the same
program to run well across different models. **Skip it** for one-shot throwaway prompts or a
single chat call — the abstraction overhead isn't worth it.

## 2. Mental Model

**DSPy is to prompting what PyTorch is to hand-derived gradients.**

In PyTorch you declare a *network* (layers + forward pass) and an *optimizer* (SGD/Adam) tunes the
weights against a *loss*. In DSPy you declare a *program* (signatures + modules + forward pass) and
an *optimizer* tunes the *prompts and demos* against a *metric*. You write the structure; the
framework figures out the parameters.

```
        you write this                         DSPy generates/tunes this
   ┌──────────────────────────┐          ┌──────────────────────────────────┐
   │ Signature: q -> answer   │          │  "You are... Given q, produce..." │
   │ Module:    ChainOfThought│  ──────► │  + 4 few-shot demos picked by the │
   │ Metric:    exact_match   │ optimizer│    optimizer to maximize metric   │
   └──────────────────────────┘          └──────────────────────────────────┘
```

The key mental shift: **prompts are a compilation target, not source code.** You stop editing
strings and start editing the *program* and the *metric*.

## 3. Key Concepts

| Term | What it is |
|------|------------|
| **Signature** | A typed I/O spec for one LM call: input fields → output fields, with descriptions. Either an inline string (`"question -> answer"`) or a `class` subclassing `dspy.Signature`. |
| **Module** | A reusable prompting strategy that *takes* a signature. `dspy.Predict` (raw), `dspy.ChainOfThought` (adds a `reasoning` field), `dspy.ReAct` (tool-using agent). Modules are composable and subclass `dspy.Module`. |
| **Adapter** | Renders a signature + demos into the concrete prompt the LM sees, and parses the response back into typed fields. The default `ChatAdapter` uses `[[ ## field ## ]]` markers. |
| **LM** | The model client, `dspy.LM("openai/gpt-4o-mini")` (LiteLLM under the hood). Set globally with `dspy.configure(lm=...)`. |
| **Optimizer / teleprompter** | Searches instructions + few-shot demos to maximize a metric. `BootstrapFewShot` (self-generates demos), `MIPROv2` (jointly optimizes instructions and demos), `BootstrapFinetune` (distills into weights). |
| **Metric** | A plain Python function `(example, prediction, trace) -> float|bool` that scores an output. This is your loss; the optimizer maximizes it. |
| **Example** | A labeled datapoint, `dspy.Example(question=..., answer=...)`, used for training/eval. `.with_inputs(...)` marks which fields are inputs. |

## 4. Setup

`pip install dspy` (the modern package; the old `dspy-ai` name is deprecated). DSPy uses
[LiteLLM](https://docs.litellm.ai/) so it talks to OpenAI, Anthropic, local Ollama/vLLM, etc.
through one interface — e.g. `dspy.LM("openai/gpt-4o-mini")` or `dspy.LM("ollama/llama3")`.

Every example below runs **offline** with `dspy.utils.DummyLM`, which returns canned, structured
responses — so the notebook executes top-to-bottom with no API key or network. The real-LLM call
shape is shown at the end, gated behind an `os.getenv` check.

In [1]:
%pip install -q dspy

import dspy
from dspy.utils.dummies import DummyLM

print("dspy version:", dspy.__version__)

Note: you may need to restart the kernel to use updated packages.


dspy version: 3.2.1


## 5. Worked Examples

### Example 1 — Signature + `Predict`: typed outputs, no prompt strings

A `Signature` declares the task; `dspy.Predict` runs it. Note that `confidence` is declared as a
`float` and DSPy parses it back into a real Python float — you describe the contract, DSPy handles
the prompt template and the parsing.

We configure a `DummyLM` with canned answers so this runs offline; with a real model you'd write
`dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))` and nothing else changes.

In [2]:
class Classify(dspy.Signature):
    """Classify the sentiment of a customer review."""
    review: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")
    confidence: float = dspy.OutputField()

# Offline stand-in for a real LM: returns these structured fields in order.
dspy.configure(lm=DummyLM([
    {"sentiment": "positive", "confidence": "0.92"},
    {"sentiment": "negative", "confidence": "0.88"},
]))

classify = dspy.Predict(Classify)

a = classify(review="Shipping was fast and the product is great!")
b = classify(review="Broke after one day, total waste of money.")

print(f"A -> {a.sentiment:8s}  confidence={a.confidence}  ({type(a.confidence).__name__})")
print(f"B -> {b.sentiment:8s}  confidence={b.confidence}")

A -> positive  confidence=0.92  (float)
B -> negative  confidence=0.88


### Example 2 — `ChainOfThought` and seeing the compiled prompt

Swapping `Predict` for `ChainOfThought` is a one-word change that inserts a `reasoning` field
*before* the answer — the model is asked to think step by step, and you get the reasoning back.
The real payoff: `dspy.inspect_history()` shows the **actual prompt DSPy built**, so you can see
that you never wrote a prompt string — the signature became the structured template below.

In [3]:
class MathQA(dspy.Signature):
    """Answer a grade-school math word problem."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

dspy.configure(lm=DummyLM([
    {"reasoning": "Roger starts with 5. 2 cans x 3 balls = 6. 5 + 6 = 11.", "answer": "11"},
]))

cot = dspy.ChainOfThought(MathQA)
r = cot(question="Roger has 5 tennis balls. He buys 2 cans of 3 balls each. How many now?")

print("answer   :", r.answer)
print("reasoning:", r.reasoning)
print("\n----- the exact prompt DSPy compiled and sent (you wrote none of it) -----")
dspy.inspect_history(n=1)

answer   : 11
reasoning: Roger starts with 5. 2 cans x 3 balls = 6. 5 + 6 = 11.

----- the exact prompt DSPy compiled and sent (you wrote none of it) -----




[2026-06-23T05:35:55.577168]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer a grade-school math word problem.


User message:

[[ ## question ## ]]
Roger has 5 tennis balls. He buys 2 cans of 3 balls each. How many now?

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
Roger starts with 5. 2 cans x 3 balls

### Example 3 — Composing modules into a custom `Module`

Real pipelines are several LM steps plus plain Python glue. You subclass `dspy.Module`, declare
sub-modules in `__init__`, and wire them in `forward`. Here a mini-RAG: step 1 builds a search
query, we retrieve (faked here), step 2 answers with chain-of-thought over the context. Because the
whole thing is one `Module`, an optimizer could later tune *both* steps' prompts jointly.

In [4]:
class BuildQuery(dspy.Signature):
    """Turn a user question into a concise search query."""
    question: str = dspy.InputField()
    search_query: str = dspy.OutputField()

class Answer(dspy.Signature):
    """Answer the question using only the retrieved context."""
    context: str = dspy.InputField()
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

def retrieve(query: str) -> str:                       # stand-in for a real vector store
    return "Paris is the capital and most populous city of France."

class RAG(dspy.Module):
    def __init__(self):
        self.make_query = dspy.Predict(BuildQuery)
        self.answer = dspy.ChainOfThought(Answer)

    def forward(self, question):
        query = self.make_query(question=question).search_query
        context = retrieve(query)
        return self.answer(context=context, question=question)

dspy.configure(lm=DummyLM([
    {"search_query": "France capital city"},                                  # step 1
    {"reasoning": "The context says Paris is the capital.", "answer": "Paris"},# step 2
]))

print("RAG answer:", RAG()(question="What's the capital of France?").answer)

RAG answer: Paris


### The real-LLM shape (gated)

Everything above used `DummyLM`. In production you swap one line — configure a real model via
LiteLLM — and the *exact same program* runs. This cell is gated on `OPENAI_API_KEY` so the notebook
still executes with no key set.

In [5]:
import os

if os.getenv("OPENAI_API_KEY"):
    dspy.configure(lm=dspy.LM("openai/gpt-4o-mini", max_tokens=200))
    qa = dspy.ChainOfThought("question -> answer")        # inline signature
    print(qa(question="What is the capital of France?").answer)
else:
    print("OPENAI_API_KEY not set - skipping the live call.")
    print("With a key, this would run the identical program against gpt-4o-mini.")

OPENAI_API_KEY not set - skipping the live call.
With a key, this would run the identical program against gpt-4o-mini.


## 6. Gotchas & Pitfalls

- **`dspy` vs `dspy-ai`.** Install `dspy` (the current package). `dspy-ai` is the deprecated alias
  and pinning it gets you stale versions.
- **It still costs tokens.** Optimizers *run your program many times* over the trainset to
  bootstrap demos and score candidates — `MIPROv2` on a few hundred examples can be hundreds of LM
  calls. Start with a tiny trainset and a cheap model, watch your spend.
- **The metric *is* the optimizer.** A sloppy metric optimizes for the wrong thing. Garbage metric
  in, garbage prompt out — invest here before reaching for a fancier optimizer.
- **Output parsing can fail.** If the model doesn't emit the `[[ ## field ## ]]` markers the adapter
  expects (common with weak/local models), you get parse errors. Use a capable model, or switch
  adapters (e.g. `JSONAdapter`).
- **Optimized state isn't auto-saved.** After `optimizer.compile(...)`, persist with
  `program.save("prog.json")` / `program.load(...)`. Otherwise you re-pay the optimization cost
  every run.
- **Global config is implicit state.** `dspy.configure(lm=...)` sets a global default. In
  concurrent code or tests, prefer the `with dspy.context(lm=...):` scope to avoid cross-talk.
- **Re-optimize when you switch models.** Demos and instructions tuned for one model aren't optimal
  for another — the whole point is that re-compiling is cheap, so do it.

## 7. When to Use vs Alternatives

| Approach | Strength | When it wins over DSPy |
|----------|----------|------------------------|
| **DSPy** | Declarative, *optimizable* multi-step pipelines; metric-driven prompt/demo tuning; model-portable. | You have an eval set and want systematic improvement, or a pipeline that must survive model swaps. |
| **Raw API / hand prompts** | Zero abstraction, full control, trivial to read. | One-shot or single-call tasks; quick prototypes; when you don't have (or want) a metric. |
| **LangChain / LlamaIndex** | Huge integration ecosystem (loaders, retrievers, tools, memory). | You mostly need *plumbing and integrations*; DSPy focuses on the prompting/optimization layer and is happy to sit inside them. |
| **Guidance / Outlines** | Constrained decoding — guarantee JSON/grammar/regex-valid output. | Hard structural guarantees matter more than prompt *optimization*. |
| **Manual few-shot + eval harness** | Total transparency; no framework. | Tiny scope, or you specifically want to own every prompt — but you're hand-doing what DSPy automates. |

**Rule of thumb:** if you can write a *metric* and have (or can make) a handful of labeled
examples, DSPy's optimizers earn their keep. If you can't define success, you don't need an
optimizer yet — start with a plain prompt.

## 8. Resources

- **Official docs** — https://dspy.ai/
- **GitHub (stanfordnlp/dspy)** — https://github.com/stanfordnlp/dspy
- **Optimizers overview** (BootstrapFewShot, MIPROv2, …) — https://dspy.ai/learn/optimization/optimizers/
- **DSPy paper** — *DSPy: Compiling Declarative LM Calls into Self-Improving Pipelines* — https://arxiv.org/abs/2310.03714
- **Programming, not prompting (concepts)** — https://dspy.ai/learn/programming/overview/